Objetivo: -   Calcule variaciones mensuales de volumen y precio

* Convertir data de raw a formato parquet
* guardar la data en interim
* escoger un producto en especifico para analizar y comenzar a armar las metricas

In [1]:
import pandas as pd
import numpy as np
import polars as pl
import pyarrow

* Obteniendo la ruta de los  archivos

In [ ]:
from pathlib import Path

BASE_DIR=Path().resolve()
DATA_PATH=BASE_DIR.parent / 'data/raw'

files = list(DATA_PATH.glob('*.xlsx'))
for f in files:
    print(f)

* funcion para leer un excel y pasarlo a polars

In [ ]:
def read_excel_polars(file_path: Path) -> pl.DataFrame:
    df_pd=pd.read_excel(file_path)
    df_pl=pl.from_pandas(df_pd)

    return df_pl

In [ ]:
dfs=[read_excel_polars(file) for file in files]
df_all=pl.concat(dfs, how='vertical_relaxed')

In [ ]:
output_path=BASE_DIR.parent / 'data/interim'
df_all.write_parquet(output_path / 'df_all.parquet')

* Analisis de data y seleccion de data
* esto con el fin de trabajar con un solo ejemplo
* un solo producto 

In [2]:
from pathlib import Path

BASE_DIR=Path().resolve()
output_path=BASE_DIR.parent / 'data/interim'
df_final=pl.scan_parquet(output_path / 'df_all.parquet')

In [ ]:
df_final.collect_schema()

In [ ]:
(
    df_final.group_by(pl.col('PARTIDA ARANCELARIA'))
    .agg(pl.col('US$ FOB').sum().alias('TOTAL_VALOR_FOB'))
    .sort('TOTAL_VALOR_FOB', descending=True)
    .collect()
).limit(20).to_pandas()

In [3]:
df_2710=df_final.filter(pl.col('PARTIDA ARANCELARIA') == 2710200012).collect()

El producto elegido esta asociado a la siguiente partida arancelaria
* Partida arancelaria 2710200012
* Descripcion arancelaria: Diesel B5, Con Un Contenido De Azufre Menor O Igual A 50 Ppm
* Producto: Diesel

Lo que se va calcular son las siguientes metricas
* Precio unitario mensual
* Volumen mensual
* Valor mensual

1. Construyes esto (nivel 1):

| periodo | precio | volumen |

In [4]:
def build_hs_monthly_base(
    df: pl.DataFrame,
    hs_code: str,
    hs_col: str = "hs_code",
    unit_col: str = "unidad_medida",
    value_col: str = "valor",
    quantity_col: str = "cantidad",
    day_col: str = "DIA",
    month_col: str = "MES",
    year_col: str = "AÑO",
) -> pl.DataFrame:
    return (
        df
        .filter(pl.col(hs_col) == hs_code)
        .filter(pl.col(unit_col).is_not_null())
        .with_columns([
            pl.col(day_col).cast(pl.Int32),
            pl.col(month_col).cast(pl.Int32),
            pl.col(year_col).cast(pl.Int32),
        ])
        .with_columns(
            pl.date(
                pl.col(year_col),
                pl.col(month_col),
                pl.col(day_col),
            ).alias("fecha")
        )
        .with_columns(
            pl.col("fecha").dt.truncate("1mo").alias("periodo")
        )
        .group_by(["periodo", hs_col, unit_col])
        .agg([
            pl.col(value_col).sum().alias("valor_total"),
            pl.col(quantity_col).sum().alias("volumen_total"),
        ])
        .with_columns(
            pl.when(pl.col("volumen_total") > 0)
            .then(pl.col("valor_total") / pl.col("volumen_total"))
            .otherwise(None)
            .alias("precio")
        )
        .select([
            "periodo",
            pl.col(hs_col).alias("hs_code"),
            pl.col(unit_col).alias("unidad_medida"),
            pl.col("volumen_total").alias("volumen"),
            "precio",
        ])
        .sort(["periodo", "unidad_medida"])
    )

In [5]:
df_2710=df_2710.with_columns(
    pl.col("PARTIDA ARANCELARIA").cast(str))

In [6]:
df_new=build_hs_monthly_base(
    df_2710,
    hs_code = "2710200012",
    hs_col = "PARTIDA ARANCELARIA",
    unit_col = "UNIDAD DE MEDIDA",
    value_col = "US$ FOB",
    quantity_col = "CANTIDAD",
    day_col = "DÍA",
    month_col = "MES",
    year_col = "AÑO",
)

In [7]:
df_new.to_pandas()

,periodo,hs_code,unidad_medida,volumen,precio
0,2025-01-01,2710200012,M3,247152.72,600.210591
1,2025-02-01,2710200012,M3,121043.01,619.224300
2,2025-03-01,2710200012,M3,191723.38,608.856242
3,2025-04-01,2710200012,M3,243913.84,566.797446
4,2025-05-01,2710200012,M3,230904.14,538.331739
5,2025-06-01,2710200012,M3,251125.15,567.081511
6,2025-07-01,2710200012,M3,247438.31,605.513328
7,2025-08-01,2710200012,M3,239018.11,601.815561
8,2025-09-01,2710200012,M3,271895.89,590.075867
9,2025-10-01,2710200012,M3,278987.61,583.093759


Construccion de funcion add_monthly_variation

In [8]:
def add_monthly_variation(df: pl.DataFrame) -> pl.DataFrame:

    if df.is_empty():
        raise ValueError("Input DataFrame is empty.")

    return (
        df.sort(["hs_code", "unidad_medida", "periodo"])
        .with_columns(
            [
                (
                    (pl.col("volumen") / pl.col("volumen").shift(1) - 1) * 100
                )
                .over(["hs_code", "unidad_medida"])
                .alias("var_pct_volumen_mensual"),
                (
                    (pl.col("precio") / pl.col("precio").shift(1) - 1) * 100
                )
                .over(["hs_code", "unidad_medida"])
                .alias("var_pct_precio_mensual"),
            ]
        )
    )

In [9]:
df_new_variation=add_monthly_variation(df_new)

In [10]:
df_new_variation

periodo,hs_code,unidad_medida,volumen,precio,var_pct_volumen_mensual,var_pct_precio_mensual
date,str,str,f64,f64,f64,f64
2025-01-01,"""2710200012""","""M3""",247152.72,600.210591,null,null
2025-02-01,"""2710200012""","""M3""",121043.01,619.2243,-51.025014,3.16784
2025-03-01,"""2710200012""","""M3""",191723.38,608.856242,58.392773,-1.674362
2025-04-01,"""2710200012""","""M3""",243913.84,566.797446,27.22175,-6.907837
2025-05-01,"""2710200012""","""M3""",230904.14,538.331739,-5.333728,-5.022201
…,…,…,…,…,…,…
2025-08-01,"""2710200012""","""M3""",239018.11,601.815561,-3.402949,-0.610683
2025-09-01,"""2710200012""","""M3""",271895.89,590.075867,13.755351,-1.950713
2025-10-01,"""2710200012""","""M3""",278987.61,583.093759,2.608248,-1.183256


[ ] 3. Volatilidad rolling

In [11]:
def add_rolling_price_volatility(
    df: pl.DataFrame,
    window: int = 6,
) -> pl.DataFrame:
    if df.is_empty():
        raise ValueError("Input DataFrame is empty.")

    if window < 2:
        raise ValueError("Window must be at least 2.")

    return (
        df.sort(["hs_code", "unidad_medida", "periodo"])
        .with_columns(
            pl.col("var_pct_precio_mensual")
            .rolling_std(window_size=window, min_samples=window)
            .over(["hs_code", "unidad_medida"])
            .alias(f"volatilidad_precio_{window}m")
        )
    )

In [13]:
df_rolling=add_rolling_price_volatility(df_new_variation,6)